In [ ]:
import hashlib
import os
import numpy as np
import pandas as pd
from google.colab import files

# 1. Upload the raw Excel file
print("Please upload 'UseCase - Airlines.xlsx':")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2. Read all sheets
df_flights = pd.read_excel(file_name, sheet_name="flights")
df_payments = pd.read_excel(file_name, sheet_name="payments")
df_bookings = pd.read_excel(file_name, sheet_name="bookings")
df_passengers = pd.read_excel(file_name, sheet_name="passengers")

print("\n   --- Starting Data Processing & Cleaning ---   ")


# A. CLEAN & TRANSFORM FLIGHTS
# Impute missing/UNKNOWN airlines using Flight ID prefixes
airline_map = {
    "AI": "Air India",
    "6F": "IndiGo",
    "SJ": "SpiceJet",
    "UK": "Vistara",
}
df_flights["prefix"] = df_flights["flight_id"].str[:2]
df_flights["airline"] = df_flights["airline"].replace("UNKNOWN", np.nan)
df_flights["airline"] = df_flights["airline"].fillna(
    df_flights["prefix"].map(airline_map)
)
df_flights.drop(columns=["prefix"], inplace=True)

# Parse timestamps and calculate exact duration in minutes

df_flights["departure_time"] = pd.to_datetime(df_flights["departure_time"])
df_flights["arrival_time"] = pd.to_datetime(df_flights["arrival_time"])

df_flights["duration_minutes"] = (
    df_flights["arrival_time"] - df_flights["departure_time"]
).dt.total_seconds() / 60.0

# Flag & fix overnight / cross-day flights (+1440 mins if arrival < departure)

df_flights["is_overnight"] = df_flights["duration_minutes"] < 0
df_flights.loc[df_flights["duration_minutes"] < 0, "duration_minutes"] += 1440

# Add Route column

df_flights["route"] = (
    df_flights["source"] + " -> " + df_flights["destination"]
)

# Flag anomalies (Durations under 30 mins or over 12 hours)

df_flights["is_anomaly"] = (df_flights["duration_minutes"] < 30) | (
    df_flights["duration_minutes"] > 720
)


# B. PII MASKING & PASSENGER CLEANING



def hash_pii(val):
    if pd.isna(val):
        return val
    return hashlib.sha256(str(val).encode()).hexdigest()[:12]


def mask_aadhaar(val):
    if pd.isna(val):
        return val
    s = str(val)
    return "XXXX-XXXX-" + s[-4:] if len(s) >= 4 else "XXXX-XXXX-0000"


# Mask the sensitive passenger fields
df_passengers["aadhaar_masked"] = df_passengers["aadhaar_id"].apply(
    mask_aadhaar
)
df_passengers["email_hashed"] = df_passengers["email"].apply(hash_pii)
df_passengers["phone_hashed"] = df_passengers["phone"].apply(hash_pii)

# Drop raw unmasked PII columns

dim_passengers = df_passengers.drop(
    columns=["email", "phone", "aadhaar_id"]
).copy()


# C. CLEAN BOOKINGS & PAYMENTS

# Fill missing status

df_bookings["status"] = df_bookings["status"].fillna("CONFIRMED")

# Hash emergency contact phone PII in bookings

df_bookings["emergency_contact_phone_hashed"] = df_bookings[
    "emergency_contact_phone"
].apply(hash_pii)
df_bookings.drop(columns=["emergency_contact_phone"], inplace=True)

# Clean payment amounts

df_payments["amount"] = pd.to_numeric(df_payments["amount"], errors="coerce")
median_amount = df_payments["amount"].median()
df_payments["amount"] = df_payments["amount"].fillna(median_amount)

# Build Fact Table by merging Bookings with Payments

fact_bookings = pd.merge(
    df_bookings,
    df_payments[["booking_id", "amount", "payment_method"]],
    on="booking_id",
    how="left",
)

# D. EXPORT CLEANED CSVs FOR POWER BI

os.makedirs("data_cleaned", exist_ok=True)

df_flights.to_csv("data_cleaned/dim_flights.csv", index=False)
dim_passengers.to_csv("data_cleaned/dim_passengers.csv", index=False)
fact_bookings.to_csv("data_cleaned/fact_bookings.csv", index=False)

print("\n--- Pipeline Executed Successfully! ---")
print(
    f"Dim Flights Records: {len(df_flights)} | Anomalies Found: {df_flights['is_anomaly'].sum()}"
)
print(f"Dim Passengers Records: {len(dim_passengers)}")
print(f"Fact Bookings Records: {len(fact_bookings)}")

# Auto-download cleaned files

for filename in [
    "dim_flights.csv",
    "dim_passengers.csv",
    "fact_bookings.csv",
]:
    files.download(f"data_cleaned/{filename}")

Please upload 'UseCase - Airlines.xlsx':


Saving UseCase - Airlines.xlsx to UseCase - Airlines.xlsx

--- Starting Data Processing & Cleaning ---

--- Pipeline Executed Successfully! ---
Dim Flights Records: 1020 | Anomalies Found: 0
Dim Passengers Records: 1039
Fact Bookings Records: 1363


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# Compute and print key business metrics
avg_duration = df_flights["duration_minutes"].mean()
route_traffic = df_flights.groupby("route").size().sort_values(ascending=False)
airline_dist = (
    df_flights.groupby("airline").size().sort_values(ascending=False)
)
total_revenue = fact_bookings["amount"].sum()


print("             BUSINESS KPI SUMMARY\n         ")

print(f"Average Flight Duration : {avg_duration:.2f} Minutes")
print(f"Total Pipeline Revenue  : ₹{total_revenue:,.2f}")
print("\nTop 5 Busiest Routes:")
print(route_traffic.head(5))
print("\nFlight Share by Airline:")
print(airline_dist)


             BUSINESS KPI SUMMARY
         
Average Flight Duration : 164.33 Minutes
Total Pipeline Revenue  : ₹8,011,258.73

Top 5 Busiest Routes:
route
BOM -> CCU    90
CCU -> DEL    74
MAA -> BLR    65
BLR -> BOM    62
HYD -> MAA    57
dtype: int64

Flight Share by Airline:
airline
IndiGo       273
Air India    260
SpiceJet     251
Vistara      236
dtype: int64
